# GRIFO: nanopore amplicon analysis

Steps, one section each:

1. **Setup**: load `config.ini` and check the paths.
2. **Read QC**: filter reads with NanoFilt, merge them per sample, report with NanoStat and MultiQC.
3. **Clustering input**: convert each sample's merged reads to a csv.
4. **Clustering**: group reads into consensus sequences and count the reads per cluster.
5. **Taxonomy and OTU table**: pool the clusters of all samples into OTUs and assign their taxonomy
   with BLAST, using the method of `haplonet.py`.
6. **Haplotype networks**: one interactive HTML page with a haplotype network per OTU.

Run the cells from top to bottom (**Kernel → Restart & Run All**). Sections 3 and 4 work from
the files in `results/qc`, so after a restart you can skip section 2 if the QC is already done.
Section 6 works from the files section 5 writes to `results/taxonomy`.

Requires NanoFilt, NanoStat, MultiQC, minimap2, spoa, vsearch, BLAST+ (`blastn`, `makeblastdb`)
and MAFFT on your PATH, numpy, pandas and scikit-learn in the notebook's Python environment, and
`haplonet.py` in the notebook folder.

## 1. Setup

In [30]:
import configparser
import csv
import gzip
import importlib
import logging
import multiprocessing
import shutil
import subprocess
import sys
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s', force=True)
log = logging.getLogger('grifo')

# All paths are relative to the notebook folder, so the notebook can be run from anywhere.
ROOT = Path.cwd()
CONFIG_FILE = ROOT / 'config.ini'

config = configparser.ConfigParser(interpolation=configparser.ExtendedInterpolation(),
                                   inline_comment_prefixes=(';', '#'))
if not config.read(CONFIG_FILE):
    raise FileNotFoundError(f'{CONFIG_FILE} not found')

path_data = ROOT / config.get('paths', 'path_data')
path_results = ROOT / config.get('paths', 'path_results', fallback='results')
QC_ROOT = path_results / 'qc'

if not path_data.is_dir():
    raise FileNotFoundError(f'Data folder not found: {path_data} (check [paths] path_data in config.ini)')


def require_tools(*tools):
    """Fail early if an external program is not on PATH."""
    missing = [t for t in tools if shutil.which(t) is None]
    if missing:
        raise RuntimeError(f'Required tool(s) not found on PATH: {", ".join(missing)}')


print('Notebook folder:', ROOT)
for section in config.sections():
    print(f'[{section}]', dict(config[section]))

Notebook folder: /Users/pascalhabluetzel/Documents/ANERIS_workshop_Crete_2026/grifo/v_0.7
[paths] {'path_data': './data', 'path_results': 'results', 'path_to_blastdb': 'MIDORI2_TOTAL_NUC_GB273_lrRNA_RAW.fasta'}
[NanoFilt] {'minlength': '100', 'maxlength': '800', 'qscore': '20'}
[clustering] {'niter': '5', 'parallel_samples': '4', 'threads': '2', 'consensus_reads': '20', 'subsample': '2000', 'sweep_partitions': '4', 'merge_threshold': '0.9', 'minimap2_options': '-k15 -w10 -p 0.9 -D', 'spoa_options': '-n -15 -g -10 -l 0 -r 0', 'seed': ''}
[read_assignment] {'min_identity': '0.85', 'min_coverage': '0.8', 'preset': 'map-ont'}
[primers] {'forward': 'GACCCTATGGAGCTTTAGAC', 'reverse': 'CCRKGGTCRCCCCAAC', 'max_mismatches': '4', 'require_both': 'no', 'search_window': '50', 'min_insert': '30', 'trim': 'yes'}
[vsearch] {'otu_identity': '1.0', 'threads': '8'}
[haplonet] {'db': 'MIDORI2_lrRNA_blast.fasta', 'blastdb': 'MIDORI2_lrRNA', 'blast_task': 'megablast', 'threads': '8', 'top': '100', 'oversam

## 2. Read QC

Filters every raw FASTQ chunk with NanoFilt, using the `[NanoFilt]` settings from `config.ini`.
Filtered chunks go to `results/qc/<sample>/filtered/`. Each sample's chunks are then merged into
`<sample>_concatenated.fastq` and `.fasta`. NanoStat and MultiQC write their report to
`results/qc/multiqc/`.

In [31]:
# --- Read QC functions ------------------------------------------------------
# External tools: NanoFilt, NanoStat, MultiQC.
FASTQ_SUFFIX = '.fastq.gz'

# Filter settings from the [NanoFilt] section of config.ini. No fallbacks:
# a missing section or key stops here instead of silently using defaults.
try:
    MIN_LENGTH = config.getint('NanoFilt', 'minlength')
    MAX_LENGTH = config.getint('NanoFilt', 'maxlength')
    MIN_QSCORE = config.getint('NanoFilt', 'qscore')  # NanoFilt -q takes whole numbers only
except (configparser.NoSectionError, configparser.NoOptionError, ValueError) as e:
    raise ValueError(f'{CONFIG_FILE}: {e}\n'
                     'Expected a [NanoFilt] section with minlength, maxlength and qscore.') from None
if MIN_LENGTH > MAX_LENGTH:
    raise ValueError(f'{CONFIG_FILE}: [NanoFilt] minlength ({MIN_LENGTH}) > maxlength ({MAX_LENGTH})')
log.info('NanoFilt settings from %s: length %d-%d bp, Q >= %s',
         CONFIG_FILE, MIN_LENGTH, MAX_LENGTH, MIN_QSCORE)


def fastq_directories(data_dir=path_data):
    """Sorted, unique directories under data_dir that hold *.fastq.gz files."""
    return sorted({p.parent for p in Path(data_dir).rglob(f'*{FASTQ_SUFFIX}')})


def sample_name(directory):
    """Sample name used to track a sample through the analysis (the directory name)."""
    return Path(directory).name


def fastq_files(directory):
    """The *.fastq.gz files directly inside directory, sorted."""
    return sorted(Path(directory).glob(f'*{FASTQ_SUFFIX}'))




def run(cmd, **kwargs):
    """Run a command; show its output in the notebook only if it fails."""
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f'{" ".join(cmd)} failed:\n{result.stdout}\n{result.stderr}')
    return result


def count_reads(path):
    """Count reads in a FASTQ file (plain or gzipped) as lines / 4.
    Counting lines that start with '@' over-counts: quality strings can start with '@'."""
    path = Path(path)
    opener = gzip.open if path.suffix == '.gz' else open
    with opener(path, 'rt') as fh:
        n_lines = sum(1 for _ in fh)
    if n_lines % 4:
        raise ValueError(f'{path}: {n_lines} lines is not a multiple of 4 (truncated file?)')
    return n_lines // 4


def count_raw_reads(directories):
    """Total raw reads per sample, summed over all FASTQ chunks in its directory."""
    return {sample_name(d): sum(count_reads(f) for f in fastq_files(d)) for d in directories}


def run_nanofilt(src, dest, min_length=MIN_LENGTH, max_length=MAX_LENGTH, min_qscore=MIN_QSCORE):
    """Stream src.fastq.gz -> NanoFilt -> dest.fastq.gz without going through a shell."""
    nanofilt_cmd = ['NanoFilt', '--length', str(min_length),
                    '--maxlength', str(max_length), '-q', str(min_qscore)]
    with open(dest, 'wb') as out:
        gunzip = subprocess.Popen(['gzip', '-dc', str(src)], stdout=subprocess.PIPE)
        nanofilt = subprocess.Popen(nanofilt_cmd, stdin=gunzip.stdout, stdout=subprocess.PIPE)
        gunzip.stdout.close()  # let gunzip get SIGPIPE if NanoFilt exits early
        gz = subprocess.Popen(['gzip', '-c'], stdin=nanofilt.stdout, stdout=out)
        nanofilt.stdout.close()
        codes = {'gzip': gz.wait(), 'NanoFilt': nanofilt.wait(), 'gunzip': gunzip.wait()}
    failed = {name: rc for name, rc in codes.items() if rc != 0}
    if failed:
        Path(dest).unlink(missing_ok=True)
        raise RuntimeError(f'Filtering {src} failed (exit codes: {failed})')


def filter_samples(directories, min_length=MIN_LENGTH, max_length=MAX_LENGTH,
                   min_qscore=MIN_QSCORE, qc_root=QC_ROOT):
    """Run NanoFilt on every FASTQ chunk of every sample."""
    for directory in directories:
        out_dir = qc_root / sample_name(directory) / 'filtered'
        out_dir.mkdir(parents=True, exist_ok=True)
        for old in fastq_files(out_dir):  # drop chunks left over from earlier runs
            old.unlink()
        for src in fastq_files(directory):
            log.info('Filtering %s', src)
            run_nanofilt(src, out_dir / src.name, min_length, max_length, min_qscore)


def merge_sample(sample, qc_root=QC_ROOT):
    """Merge a sample's filtered chunks into one FASTQ and one FASTA in a single pass.
    Returns the number of reads written."""
    sample_dir = qc_root / sample
    n_reads = 0
    with open(sample_dir / f'{sample}_concatenated.fastq', 'w') as fq, \
         open(sample_dir / f'{sample}_concatenated.fasta', 'w') as fa:
        for chunk in fastq_files(sample_dir / 'filtered'):
            with gzip.open(chunk, 'rt') as fh:
                # zip(*[fh] * 4) yields one 4-line FASTQ record at a time
                for header, seq, plus, qual in zip(*[fh] * 4):
                    fq.write(header + seq + plus + qual)
                    fa.write('>' + header[1:] + seq)
                    n_reads += 1
    return n_reads


def merge_samples(samples, qc_root=QC_ROOT):
    """Merge every sample; returns filtered read counts per sample."""
    return {s: merge_sample(s, qc_root) for s in samples}


def run_stats(samples, qc_root=QC_ROOT):
    """Run NanoStat per sample, then MultiQC over all reports."""
    report_dir = qc_root / 'multiqc'
    report_dir.mkdir(parents=True, exist_ok=True)
    for s in samples:
        log.info('NanoStat on %s', s)
        run(['NanoStat', '--fastq', str(qc_root / s / f'{s}_concatenated.fastq'),
             '--outdir', str(report_dir), '--name', f'{s}_statreports'])
    log.info('Running MultiQC')
    run(['multiqc', '.', '--force'], cwd=report_dir)
    return report_dir / 'multiqc_report.html'

2026-11-11 11:45:44,561 INFO NanoFilt settings from /Users/pascalhabluetzel/Documents/ANERIS_workshop_Crete_2026/grifo/v_0.7/config.ini: length 100-800 bp, Q >= 20


In [25]:
# --- Run read QC -----------------------------------------------------------
require_tools('gzip', 'NanoFilt', 'NanoStat', 'multiqc')

directories = fastq_directories(path_data)
if not directories:
    raise FileNotFoundError(f'No *{FASTQ_SUFFIX} files found under {path_data}')
samples = [sample_name(d) for d in directories]

raw_counts = count_raw_reads(directories)
filter_samples(directories)  # uses [NanoFilt] from config.ini
filtered_counts = merge_samples(samples)
report = run_stats(samples)

print(f'\nNanoFilt: length {MIN_LENGTH}-{MAX_LENGTH} bp, Q >= {MIN_QSCORE}')
print(f'\n{"sample":<20}{"raw":>10}{"filtered":>10}{"kept":>8}')
for s in samples:
    raw, kept = raw_counts[s], filtered_counts[s]
    pct = 100 * kept / raw if raw else 0
    print(f'{s:<20}{raw:>10}{kept:>10}{pct:>7.1f}%')
print(f'\nMultiQC report: {report}')

2026-11-11 11:41:35,172 INFO Filtering /Users/pascalhabluetzel/Documents/ANERIS_workshop_Crete_2026/grifo/v_0.7/data/barcode01/fastq_runid_28a8233a-3f5d-4b3d-974c-ee46b052711f_0_0.fastq.gz
2026-11-11 11:41:35,615 INFO Filtering /Users/pascalhabluetzel/Documents/ANERIS_workshop_Crete_2026/grifo/v_0.7/data/barcode01/fastq_runid_28a8233a-3f5d-4b3d-974c-ee46b052711f_10_0.fastq.gz
2026-11-11 11:41:35,867 INFO Filtering /Users/pascalhabluetzel/Documents/ANERIS_workshop_Crete_2026/grifo/v_0.7/data/barcode01/fastq_runid_28a8233a-3f5d-4b3d-974c-ee46b052711f_11_0.fastq.gz
2026-11-11 11:41:36,123 INFO Filtering /Users/pascalhabluetzel/Documents/ANERIS_workshop_Crete_2026/grifo/v_0.7/data/barcode01/fastq_runid_28a8233a-3f5d-4b3d-974c-ee46b052711f_12_0.fastq.gz
2026-11-11 11:41:36,373 INFO Filtering /Users/pascalhabluetzel/Documents/ANERIS_workshop_Crete_2026/grifo/v_0.7/data/barcode01/fastq_runid_28a8233a-3f5d-4b3d-974c-ee46b052711f_13_0.fastq.gz
2026-11-11 11:41:36,628 INFO Filtering /Users/pasca


NanoFilt: length 100-800 bp, Q >= 20

sample                     raw  filtered    kept
barcode01                17999      9475   52.6%
barcode02               435086    313833   72.1%
barcode03               248586    180251   72.5%
barcode04                18358      5922   32.3%

MultiQC report: /Users/pascalhabluetzel/Documents/ANERIS_workshop_Crete_2026/grifo/v_0.7/results/qc/multiqc/multiqc_report.html


## 3. Clustering input

Converts each sample's `<sample>_concatenated.fasta` to `<sample>_concatenated.csv`, with columns
`id, sequence`. The id is the read name, i.e. the first word of the FASTA header. Samples are
found from the files in `results/qc`, so this section doesn't depend on variables from section 2.

In [32]:
def qc_samples(qc_root=QC_ROOT):
    """Samples in results/qc that have a merged FASTA from the read QC."""
    return sorted(d.name for d in qc_root.iterdir()
                  if (d / f'{d.name}_concatenated.fasta').is_file())


def fasta2csv(sample, qc_root=QC_ROOT):
    """Write <sample>_concatenated.csv [id, sequence] from <sample>_concatenated.fasta.
    Handles sequences split over several lines; empty records are skipped.
    Returns (reads written, empty records skipped)."""
    sample_dir = qc_root / sample
    n_written = n_empty = 0
    with open(sample_dir / f'{sample}_concatenated.fasta') as fin, \
         open(sample_dir / f'{sample}_concatenated.csv', 'w', newline='') as fout:
        writer = csv.writer(fout)
        writer.writerow(['id', 'sequence'])
        read_id, seq = None, []
        for line in list(fin) + ['>']:          # the final '>' flushes the last record
            line = line.strip()
            if line.startswith('>'):
                if read_id is not None:
                    if seq:
                        writer.writerow([read_id, ''.join(seq)])
                        n_written += 1
                    else:
                        n_empty += 1
                words = line[1:].split()
                read_id = words[0] if words else f'read{n_written + n_empty}'
                seq = []
            elif line:
                seq.append(line.upper())
    return n_written, n_empty


samples = qc_samples()
if not samples:
    raise FileNotFoundError(f'No *_concatenated.fasta files in {QC_ROOT}: run section 2 first')

for s in samples:
    n_written, n_empty = fasta2csv(s)
    note = f' ({n_empty} empty records skipped)' if n_empty else ''
    print(f'{s}: {n_written} reads{note}')

barcode01: 9475 reads
barcode02: 313833 reads
barcode03: 180251 reads
barcode04: 5922 reads


## 4. Clustering

The next cell saves the clustering code as `clustering_tools.py` next to the notebook. It has to be
a file because the samples run in parallel worker processes, and those import their code from a file.
After editing that cell, run it again; the cell after it reloads the file.

For each sample this writes to `results/qc/<sample>/`:

| file | contents |
|---|---|
| `<sample>_clusters.csv` | one consensus sequence per cluster (`id, sequence, split`) |
| `<sample>_read_assignments.csv` | the cluster each read was assigned to, with identity and coverage |
| `<sample>_cluster_counts.csv` | reads per cluster, plus an `unassigned` row |
| `<sample>_clustering.log` | progress log of the clustering |

The number of iterations comes from `[clustering] niter` in `config.ini`.

In [33]:
%%writefile clustering_tools.py
# ============================================================================
# Clustering of amplicon reads into consensus sequences + read assignment
#
#   cluster()      groups reads into clusters and returns one consensus
#                  sequence per cluster
#   assign_reads() assigns every read to its best-matching cluster and
#                  counts reads per cluster
#
# Running this cell saves it as clustering_tools.py next to the notebook;
# the next cell imports it. Requires minimap2 and spoa on your PATH.
# Works with pandas 1.x and 2.x, Python >= 3.8.
#
# Input : csv with columns [id, sequence] (other columns ignored), fasta or
#         fastq. Output: csv [id, sequence, split]; ids are cluster0, ...
#
# License: the clustering algorithm is a port of code that is
# Copyright (C) 2020 Bilgenur Baloglu, Centre for Biodiversity Genomics,
# University of Guelph, distributed under the GNU Lesser General Public
# License version 3 (https://www.gnu.org/licenses/lgpl-3.0.html). This code
# is distributed under the same license.
# ============================================================================

import os
import re
import shutil
import logging
import gzip
import tempfile
import warnings
import subprocess

import numpy as np
import pandas as pd
import sklearn.cluster

logger = logging.getLogger('cluster')

# default clustering parameters
DEFAULTS = dict(
    n_iter=10,                           # clustering iterations
    csize=20,                            # reads per cluster used for consensus
    N=2000,                              # subsample size for pairwise distances
    th_s=4,                              # k-means partitions in the sweep step
    th_m=0.9,                            # distance threshold for merging clusters
    pw_config='-k15 -w10 -p 0.9 -D',     # minimap2 options
    msa_config='-n -15 -g -10 -l 0 -r 0' # spoa options
)


#############################################################################
# Main functions
#############################################################################

def cluster(input_file, output_file=None, n_iter=DEFAULTS['n_iter'],
            csize=DEFAULTS['csize'], N=DEFAULTS['N'], th_s=DEFAULTS['th_s'],
            th_m=DEFAULTS['th_m'], pw_config=DEFAULTS['pw_config'],
            msa_config=DEFAULTS['msa_config'], init_centers=None,
            minimap2='minimap2', spoa='spoa', threads=3, seed=None,
            workspace=None, log_file=None):
    '''
    Cluster sequences into cluster-center consensus sequences.

    input_file   = csv with columns [id, sequence], fasta or fastq
    output_file  = csv to write [id, sequence, split]; None to skip writing
    n_iter       = clustering iterations
    csize        = sequences per cluster used to build the spoa consensus
    N            = subsample size used for pairwise distances
    th_s         = number of k-means partitions in the sweep step
    th_m         = OPTICS eps threshold for merging similar clusters
    pw_config    = minimap2 options for read-vs-center alignment
    msa_config   = spoa options for consensus building
    init_centers = optional csv/fasta of starting cluster centers
    minimap2, spoa = executable names or paths
    threads      = threads per minimap2 call, and max parallel spoa jobs
    seed         = integer for reproducible runs (otherwise results vary
                   slightly between runs, as reads are sampled at random)
    workspace    = folder for temporary files (default: temp folder next to
                   output_file, removed afterwards)
    log_file     = optional log file path
    returns pandas DataFrame of cluster centers [id, sequence, split]
    '''
    _check_tool(minimap2)
    _check_tool(spoa)
    if seed is not None:
        np.random.seed(seed)
    handler = None
    if log_file:
        handler = logging.FileHandler(log_file, mode='w')
        handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s: %(message)s'))
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)

    out_dir = os.path.dirname(os.path.abspath(output_file)) if output_file else os.getcwd()
    own_workspace = workspace is None
    if own_workspace:
        workspace = tempfile.mkdtemp(prefix='cluster_', dir=out_dir)
    else:
        os.makedirs(workspace, exist_ok=True)

    tools = dict(minimap2=minimap2, spoa=spoa, threads=int(threads),
                 seed=seed, workspace=workspace)
    try:
        logger.info('Loading %s', input_file)
        df = _load_sequences(input_file)
        df['id'] = ['read' + str(i) for i in range(len(df))]
        # keep only reads with both primers if that info exists
        if 'fwd_primer_seq' in df.columns and 'rev_primer_seq' in df.columns:
            df = df[df['fwd_primer_seq'].notna() & df['rev_primer_seq'].notna()]
        df = df[['id', 'sequence']].reset_index(drop=True)
        logger.info('%d sequences to cluster', len(df))

        df_d = []
        if init_centers:
            df_d = _load_sequences(init_centers)[['id', 'sequence']]

        data = perform_cluster(df, df_d, max_iter=int(n_iter), csize=int(csize),
                               N=int(N), th_s=int(th_s), th_m=float(th_m),
                               pw_config=pw_config, msa_config=msa_config,
                               tools=tools)
        data = data[['id', 'sequence', 'split']].reset_index(drop=True)
        if output_file:
            data.to_csv(output_file, index=False)
            logger.info('Wrote %d clusters to %s', len(data), output_file)
        return data
    finally:
        if own_workspace:
            shutil.rmtree(workspace, ignore_errors=True)
        if handler is not None:
            logger.removeHandler(handler)
            handler.close()


def assign_reads(reads_file, clusters_file, assignments_file='auto',
                  counts_file='auto', min_identity=0.85, min_coverage=0.8,
                  minimap2='minimap2', threads=3, preset='map-ont', workspace=None):
    '''
    Assign every read to its best-matching cluster center and count reads per
    cluster. cluster() returns centers only; this adds read membership.

    reads_file    = the reads that were clustered: csv [id, sequence], fasta or
                    fastq (optionally .gz)
    clusters_file = output of cluster(): csv [id, sequence, ...] or fasta
    assignments_file = csv for the per-read table; 'auto' derives the name from
                    clusters_file (X_clusters.csv -> X_read_assignments.csv);
                    None to skip writing
    counts_file   = csv for reads per cluster; 'auto' -> X_cluster_counts.csv;
                    None to skip writing
    min_identity  = minimum alignment identity (matching bases / alignment
                    length) for a read to be assigned. Raw nanopore reads vs
                    a consensus typically score 0.85-0.97 depending on chemistry
    min_coverage  = minimum fraction of the read covered by the alignment
    preset        = minimap2 preset (-x)

    Each read goes to the cluster with the highest alignment score (AS).
    'ambiguous' = True when another cluster scored exactly as well; the read
    is still assigned (to the first such cluster) but you may want to
    exclude or split these for closely related clusters.

    returns (assignments, counts) as pandas DataFrames
      assignments: read_id, cluster, identity, coverage, AS, ambiguous, status
                   status is 'assigned', 'below_threshold' or 'no_hit'
      counts:      cluster, n_reads, n_ambiguous, plus an 'unassigned' row
    '''
    _check_tool(minimap2)
    reads = _load_sequences(reads_file)
    # keep the first word of the header as the read id (drops '>' and the
    # 'runid=... ch=...' fields that nanopore headers carry)
    reads['read_id'] = [str(i).lstrip('>').split()[0] if str(i).strip() else 'read%d' % k
                        for k, i in enumerate(reads['id'])]
    clusters = _load_sequences(clusters_file)[['id', 'sequence']]
    logger.info('assign_reads: %d reads vs %d clusters', len(reads), len(clusters))

    own_workspace = workspace is None
    if own_workspace:
        base_dir = os.path.dirname(os.path.abspath(clusters_file))
        workspace = tempfile.mkdtemp(prefix='assign_reads_', dir=base_dir)
    tools = dict(minimap2=minimap2, threads=int(threads), workspace=workspace)
    try:
        aln = run_minimap2(reads[['read_id', 'sequence']].rename(columns={'read_id': 'id'}),
                           clusters, config='-x ' + preset, cigar=True, tools=tools)
    finally:
        if own_workspace:
            shutil.rmtree(workspace, ignore_errors=True)

    if len(aln) > 0:
        aln['identity'] = aln['match'] / aln['tot']
        aln['coverage'] = (aln['q_end'] - aln['q_start']) / aln['q_len']
        # best alignment per read; detect ties with a different cluster
        aln = aln.sort_values(['query_id', 'AS', 'identity'], ascending=[True, False, False])
        best = aln.drop_duplicates('query_id', keep='first').set_index('query_id')
        top_as = best['AS'].reindex(aln['query_id']).values
        ties = aln[(aln['AS'].values == top_as)].groupby('query_id')['database_id'].nunique()
        best['ambiguous'] = ties.reindex(best.index).fillna(1).values > 1
    else:
        best = pd.DataFrame(columns=['database_id', 'identity', 'coverage', 'AS', 'ambiguous'])

    out = pd.DataFrame({'read_id': reads['read_id'].values})
    b = best.reindex(out['read_id'])
    out['cluster'] = b['database_id'].values
    out['identity'] = b['identity'].values.astype(float).round(4)
    out['coverage'] = b['coverage'].values.astype(float).round(4)
    out['AS'] = b['AS'].values
    out['ambiguous'] = b['ambiguous'].fillna(False).values.astype(bool)
    hit = out['cluster'].notna().values
    ok = hit & (out['identity'].values >= min_identity) & (out['coverage'].values >= min_coverage)
    out['status'] = np.where(ok, 'assigned', np.where(hit, 'below_threshold', 'no_hit'))
    out.loc[~ok, 'cluster'] = None
    out.loc[~ok, 'ambiguous'] = False

    assigned = out[out['status'] == 'assigned']
    counts = pd.DataFrame({'cluster': clusters['id'].values})
    counts['n_reads'] = counts['cluster'].map(assigned['cluster'].value_counts()).fillna(0).astype(int)
    counts['n_ambiguous'] = counts['cluster'].map(
        assigned[assigned['ambiguous']]['cluster'].value_counts()).fillna(0).astype(int)
    counts = counts.sort_values('n_reads', ascending=False).reset_index(drop=True)
    counts = pd.concat([counts, pd.DataFrame({'cluster': ['unassigned'],
                                              'n_reads': [int((~ok).sum())],
                                              'n_ambiguous': [0]})], ignore_index=True)

    stem = re.sub(r'(_clusters)?\.(csv|fa|fasta)(\.gz)?$', '', clusters_file)
    if assignments_file == 'auto':
        assignments_file = stem + '_read_assignments.csv'
    if counts_file == 'auto':
        counts_file = stem + '_cluster_counts.csv'
    if assignments_file:
        out.to_csv(assignments_file, index=False)
    if counts_file:
        counts.to_csv(counts_file, index=False)
    logger.info('assign_reads: %d assigned (%d ambiguous), %d below threshold, %d no hit',
                ok.sum(), out['ambiguous'].sum(), (out['status'] == 'below_threshold').sum(),
                (out['status'] == 'no_hit').sum())
    return out, counts


#############################################################################
# Clustering logic
#############################################################################

def perform_cluster(df_q, df_d, max_iter, csize, N, th_s, th_m, pw_config,
                    msa_config, tools):
    '''Refine cluster centers: initial sampling, then split / sweep / merge.'''
    if len(df_d) > 0:
        df_c = df_d.copy()
    else:
        df_c = cluster_sample(df_q, N=5, th=0.8, rsize=1, config=pw_config, tools=tools)
    df_c['split'] = True
    for k in range(max_iter):
        logger.info('perform_cluster: iter = %d/%d', k, max_iter)
        df_c = cluster_split(df_q, df_c, N, csize, pw_config, msa_config, tools)
        df_c = cluster_sweep(df_q, df_c, th_s, N, csize, pw_config, msa_config, tools)
        df_c = cluster_merge(df_q, df_c, th_m, 100, csize, pw_config, msa_config, tools)
    # final refinement
    df_c['split'] = True
    df_c = cluster_split(df_q, df_c, 200, csize, pw_config, msa_config, tools)
    df_c = cluster_merge(df_q, df_c, th_m, 100, csize, pw_config, msa_config, tools)
    logger.info('perform_cluster: complete, %d clusters', len(df_c))
    return df_c


def cluster_sample(df_q, N=5, th=0.8, rsize=1, config='', tools=None):
    '''Pick up to N rough, mutually dissimilar starting centers.'''
    qry = np.unique(df_q['id'].astype(str))
    if len(qry) == 0:
        logger.info('cluster_sample: nothing to cluster')
        return pd.DataFrame([], columns=['id', 'sequence'])
    # seed with a random read
    ridx = np.random.permutation(len(qry))
    df_c = df_q[df_q['id'].isin(qry[ridx[:rsize]])][['id', 'sequence']]
    db = df_c
    while N > len(df_c):
        logger.info('cluster_sample: qlen=%d progress=%d/%d', len(qry), len(df_c), N)
        seq = df_q[df_q['id'].isin(qry)]
        out = run_minimap2(seq, db, config=config, cigar=True, tools=tools)
        out = out[out['match_score'] > th]
        qry = set(qry) - set(out['query_id'].astype(str)) - set(df_c['id'].astype(str))
        if len(qry) == 0:
            break
        qry = np.array(sorted(qry))
        ridx = np.random.permutation(len(qry))
        db = df_q[df_q['id'].isin(qry[ridx[:rsize]])][['id', 'sequence']]
        df_c = _concat([df_c, db])
    df_c = df_c.reset_index(drop=True)
    df_c['id'] = ['cluster' + str(i) for i in range(len(df_c))]
    return df_c


def cluster_eval(df_q, df_c, pw_config, tools, cigar=True):
    '''Align reads to centers; drop centers that attract no reads.'''
    df_align = run_minimap2(df_q, df_c, config=pw_config + ' --for-only',
                            cigar=cigar, tools=tools)
    cols = np.unique(df_align['database_id'])
    df_c = df_c[df_c['id'].isin(cols)]
    logger.info('cluster_eval: number of clusters = %d', len(df_c))
    return df_c, df_align


def cluster_split(df_q, df_c, N, csize, pw_config, msa_config, tools):
    '''Re-cluster the reads of each center marked split=True.'''
    df_c, df_align = cluster_eval(df_q, df_c, pw_config, tools, cigar=False)
    split = df_c['split'].astype(bool).values
    squeue = df_c['id'].values[split]
    df_c = df_c[~split]
    if len(df_align) > 0:
        df_align['weight'] = cluster_weight(df_align, metric='s1')
    for i, qout in enumerate(squeue):
        logger.info('cluster_split: splitting on cid=%s %d/%d', qout, i, len(squeue))
        qlist = cluster_subsample(df_align, [qout], N=N, mode='semi', metric='s1')
        qlist = df_q[df_q['id'].isin(qlist)]
        cout = cluster_compute(qlist, csize, pw_config=pw_config,
                               msa_config=msa_config, tools=tools)
        cout['split'] = len(cout) > 1
        df_c = _concat([df_c, cout[['id', 'sequence', 'split']]])
    return _rename_clusters(df_c)


def cluster_sweep(df_q, df_c, th_s, N, csize, pw_config, msa_config, tools):
    '''Find new centers among reads that align poorly to existing centers.'''
    df_c, df_align = cluster_eval(df_q, df_c, pw_config, tools, cigar=False)
    covered = set()
    if len(df_align) > 0:
        # normalise chaining score s1 by the best s1 for that center
        df_a = get_best(df_align, ['database_id'], metric='s1').rename(columns={'s1': 'b1'})
        df_align = df_align.merge(df_a[['database_id', 'b1']], on='database_id', how='left')
        df_align['m1'] = df_align['s1'] / df_align['b1']
        df_align = get_best(df_align, ['query_id'], metric='m1')
        df_align = df_align.rename(columns={'query_id': 'id'})
        if len(df_align) >= th_s:
            labels = cluster_kmeans_1d(df_align['m1'].values.astype(float), n_clusters=th_s,
                                       seed=tools['seed'])
            covered = set(df_align['id'].astype(str).values[labels != 0])  # 0 = worst partition
        else:
            covered = set(df_align['id'].astype(str))
    uncover = set(df_q['id'].astype(str)) - covered
    uncover = df_q[df_q['id'].isin(list(uncover))]
    logger.info('cluster_sweep: uncovered %d/%d', len(uncover), len(df_q))
    ridx = np.random.permutation(len(uncover))
    x = uncover.iloc[ridx[:N]]
    cout = cluster_compute(x, csize, pw_config=pw_config, msa_config=msa_config, tools=tools)
    cout['split'] = True
    df_c = _concat([df_c, cout[['id', 'sequence', 'split']]])
    return _rename_clusters(df_c)


def cluster_merge(df_q, df_c, th_m, N, csize, pw_config, msa_config, tools):
    '''Merge centers whose read-alignment profiles are similar.'''
    df_c, df_align = cluster_eval(df_q, df_c, pw_config, tools, cigar=True)
    if len(df_align) == 0:
        return _rename_clusters(df_c)
    # cosine distance between centers, based on AS of reads against each center
    vec = get_feature_vector(df_align[['query_id', 'database_id', 'AS']])
    cols = vec.columns[vec.columns != 'id']
    x = vec[cols].values
    cosim = pd.DataFrame(dist_cosine(x.T, x.T), columns=cols)
    cosim['id'] = cols
    df_clst = cluster_optics(cosim, min_samples=2, max_eps=th_m)
    c = (df_clst['cluster_id'] == -1).values
    df_c = df_c[df_c['id'].isin(df_clst['id'].values[c])]
    mqueue = [list(df_clst[df_clst['cluster_id'] == cid]['id'].values)
              for cid in np.unique(df_clst['cluster_id'].values[~c])]
    df_align['weight'] = cluster_weight(df_align, metric='AS')
    logger.info('cluster_merge: %d/%d clusters to merge', np.sum(~c), len(df_clst))
    for i, mout in enumerate(mqueue):
        logger.info('cluster_merge: merging %d clusters, %d/%d', len(mout), i, len(mqueue))
        qlist = cluster_subsample(df_align, mout, N=N, mode='sorted', metric='AS')
        qlist = df_q[df_q['id'].isin(qlist)]
        cout = cluster_compute(qlist, csize, pw_config=pw_config,
                               msa_config=msa_config, tools=tools)
        cout['split'] = True
        df_c = _concat([df_c, cout[['id', 'sequence', 'split']]])
    return _rename_clusters(df_c)


def cluster_weight(df_align, metric='s1'):
    '''Weight of each alignment relative to the read's best alignment.'''
    df_a = get_best(df_align, ['query_id'], metric=metric).rename(columns={metric: 'b1'})
    merged = df_align[['query_id', metric]].merge(df_a[['query_id', 'b1']],
                                                  on='query_id', how='left')
    w = merged[metric].values / (merged[metric].values + merged['b1'].values)
    return w


def cluster_subsample(df_align, dlist, N=100, mode='sorted', metric='AS'):
    '''Pick reads belonging to the given centers, favouring strong alignments.'''
    if len(df_align) == 0:
        return np.array([])
    df_align = df_align.sort_values(by=['database_id', metric])[::-1]  # best first
    x = []
    for d in dlist:
        sub = df_align[df_align['database_id'] == d]
        if mode == 'rand':
            v = sub['query_id'].values
            qlist = v[np.random.permutation(len(v))[:N]]
        else:
            q = sub['query_id'].values
            w = sub['weight'].values.astype(float).copy()
            w[w >= 0.5] = 1
            keep = w > np.random.rand(len(w))  # weighted random acceptance
            q = q[keep]
            if mode == 'semi':
                qlist = q[np.random.permutation(len(q))[:N]]
            else:  # sorted
                qlist = q[:N]
        x += list(qlist)
    return np.unique(x)


def cluster_compute(df_q, csize=20, metric='similarity', pw_config='-k15 -w10 -D',
                    msa_config='-l 0 -r 0', tools=None):
    '''
    Cluster a set of reads: all-vs-all minimap2 -> distance matrix -> OPTICS ->
    spoa consensus per cluster. Returns [id, sequence] of the new centers.
    '''
    empty = pd.DataFrame([], columns=['id', 'sequence'])
    df_q = df_q.copy()
    df_q['id'] = df_q['id'].astype(str)
    if len(df_q) < 2:
        logger.warning('cluster_compute: sample size=%d, too small to do anything', len(df_q))
        return empty
    logger.info('cluster_compute: computing pairwise distance matrix (%d reads)', len(df_q))
    df = run_minimap2(df_q, df_q, config=pw_config + ' --dual=no --for-only',
                      cigar=True, tools=tools)
    if len(df) == 0:
        logger.warning('cluster_compute: not enough samples to cluster')
        return empty
    df = get_best(df, ['query_id', 'database_id'], 'AS')
    df = get_feature_vector(df[['query_id', 'database_id', metric]], symmetric=True)
    df = get_symmetric_matrix(df)
    # similarity -> distance
    cols = df.columns[df.columns != 'id']
    dist = 1 - df[cols].values.astype(float)
    np.fill_diagonal(dist, 0)
    dist_df = pd.DataFrame(dist, columns=cols)
    dist_df['id'] = df['id'].values

    logger.info('cluster_compute: running optics')
    df_clst = cluster_optics(dist_df)
    df_clst = df_clst.merge(df_q, on='id', how='left')
    df_i = df_clst[df_clst['cluster_id'] > -1]
    if len(df_i) == 0:
        logger.warning('cluster_compute: no clusters found')
        return empty
    din = [df_i[df_i['cluster_id'] == cid].iloc[:csize] for cid in np.unique(df_i['cluster_id'])]
    return cluster_spoa_merge(pd.concat(din), config=msa_config, tools=tools)


def cluster_spoa_merge(df, config='-l 0 -r 0', tools=None):
    '''One spoa consensus per cluster_id. Returns [id, sequence].'''
    work = tempfile.mkdtemp(prefix='spoa_', dir=tools['workspace'])
    try:
        df = df.sort_values(by=['cluster_id', 'id'])
        infiles = []
        for cid, grp in df.groupby('cluster_id', sort=True):
            fname = os.path.join(work, str(cid) + '.fa')
            write_fasta(fname, grp[['id', 'sequence']].values)
            infiles.append((cid, fname))
        # run spoa jobs, `threads` at a time
        cmd_base = [tools['spoa']] + config.split()
        nproc = max(1, tools['threads'])
        for start in range(0, len(infiles), nproc):
            procs = []
            for cid, fname in infiles[start:start + nproc]:
                fout = open(fname + '.out', 'w')
                procs.append((subprocess.Popen(cmd_base + [fname], stdout=fout,
                                               stderr=subprocess.PIPE), fout, fname))
            for p, fout, fname in procs:
                _, err = p.communicate()
                fout.close()
                if p.returncode != 0:
                    raise RuntimeError('spoa failed on %s: %s' % (fname, err.decode(errors='replace')))
        data = []
        for cid, fname in infiles:
            seq = read_spoa(fname + '.out')
            if len(seq) == 0:
                # empty consensus sequences would show up in BLAST as
                # "Sequence contains no data"; drop them
                logger.warning('cluster_spoa_merge: empty consensus for cluster %s, dropped', cid)
                continue
            data.append(['cluster' + str(cid), seq])
        return pd.DataFrame(data, columns=['id', 'sequence'])
    finally:
        shutil.rmtree(work, ignore_errors=True)


#############################################################################
# Clustering primitives
#############################################################################

def get_best(df, col, metric='AS', stat='idxmax'):
    '''Row with the best `metric` for each group in `col`.'''
    df = df.reset_index(drop=True)
    idx = df.groupby(by=col)[metric].agg(stat).values
    return df.iloc[idx].reset_index(drop=True)


def get_feature_vector(df, symmetric=False):
    '''
    Pivot [query_id, database_id, value] into a matrix.
    returns dataframe with one column per database_id (sorted) plus 'id'
    '''
    q = df.iloc[:, 0].values
    d = df.iloc[:, 1].values
    m = df.iloc[:, 2].values.astype(float)
    if symmetric:
        qlist = np.unique(np.concatenate([q, d]))
        dlist = qlist
    else:
        qlist = np.unique(q)
        dlist = np.unique(d)
    q_dct = {k: i for i, k in enumerate(qlist)}
    d_dct = {k: i for i, k in enumerate(dlist)}
    v = np.zeros([len(qlist), len(dlist)])
    for qi, di, mi in zip(q, d, m):
        v[q_dct[qi], d_dct[di]] = mi
    out = pd.DataFrame(v, columns=list(dlist))
    out['id'] = list(qlist)
    return out


def get_symmetric_matrix(df):
    '''
    Make a square similarity matrix symmetric. For each pair the smaller
    non-zero value is kept (0 = no alignment).
    '''
    cols = df.columns[df.columns != 'id']
    a = df[cols].values.astype(float)
    at = a.T
    sym = np.where(a == 0, at, np.where(at == 0, a, np.minimum(a, at)))
    out = pd.DataFrame(sym, columns=cols)
    out['id'] = df['id'].values
    return out


def dist_cosine(v1, v2):
    '''Cosine distance between rows of v1 and rows of v2 (v1 == v2 here).'''
    B = np.matmul(v1, v2.T)
    norms = np.sqrt(np.diag(B)).reshape(-1, 1)
    D = B / np.matmul(norms, norms.T)
    D[D > 1] = 1  # numerical imprecision
    return 1 - D


def cluster_optics(df, min_samples=None, max_eps=None, xi=0.05,
                   cluster_method='dbscan', min_cluster_size=None):
    '''
    OPTICS on a precomputed distance matrix (df = [d_1..d_n, id]).
    If min_samples is None, search for the value giving the most clusters with
    the fewest outliers.
    returns dataframe [id, cluster_id, ordering, reachability] in OPTICS order
    '''
    if len(df) < 3:
        logger.warning('cluster_optics: sample size too small, each item is its own cluster')
        return pd.DataFrame({'id': df['id'].values, 'cluster_id': np.arange(len(df)),
                             'ordering': np.arange(len(df)), 'reachability': -1.0})
    cols = df.columns[df.columns != 'id']
    X = df[cols].values.astype(float)
    if max_eps is None:
        z = np.abs(X)
        max_eps = (np.max(z) + np.min(z)) / 2
    if min_samples is None or min_samples > len(df):
        min_samples = np.ceil(len(df) / 2)
        max_iter = 100
    else:
        max_iter = 1

    prev_nout, prev_nclust, prev_min = len(df), 0, min_samples * 2
    best_clust, best_nclust, best_nout = None, 0, prev_nout
    delta = min_samples / 2
    tmp = prev_min
    for i in range(max_iter):
        min_samples = int(min(min_samples, len(df)))  # sklearn needs min_samples <= n
        clust = sklearn.cluster.OPTICS(min_samples=min_samples, min_cluster_size=min_cluster_size,
                                       xi=xi, max_eps=max_eps, cluster_method=cluster_method,
                                       metric='precomputed')
        with warnings.catch_warnings():
            # the min_samples search deliberately tries settings where every
            # point is an outlier; sklearn warns about those, which is expected
            warnings.simplefilter('ignore', UserWarning)
            clust.fit(X)
        nout = int(np.sum(clust.labels_ == -1))
        nclust = int(np.max(clust.labels_) + 1)
        logger.debug('cluster_optics: iter=%d min_samples=%d clusters=%d outliers=%d',
                     i, min_samples, nclust, nout)
        if nclust > prev_nclust or (nclust == prev_nclust and nout <= prev_nout):
            tmp = min_samples
            delta = int((prev_min - min_samples) / 2)
            min_samples -= delta
        elif nclust < prev_nclust:
            tmp = min_samples
            delta = int((prev_min + min_samples) / 2) - min_samples
            min_samples = prev_min + delta
        if best_clust is None or nclust > best_nclust or (nclust == best_nclust and nout <= best_nout):
            if np.min(clust.reachability_) >= 0:
                best_clust, best_nclust, best_nout = clust, nclust, nout
        if delta == 0 or min_samples <= 2 or nout == 0:
            break
        prev_nclust, prev_nout, prev_min = nclust, nout, tmp

    s = best_clust.ordering_
    reach = best_clust.reachability_[s].copy()
    finite = reach[np.isfinite(reach)]
    reach[~np.isfinite(reach)] = np.max(finite) if len(finite) else -1
    return pd.DataFrame({'id': df['id'].values[s], 'cluster_id': best_clust.labels_[s].astype(int),
                         'ordering': np.arange(len(s)), 'reachability': reach.astype(float)})


def cluster_kmeans_1d(values, n_clusters, seed=None):
    '''
    k-means on a 1-D array; labels are re-numbered so 0 is the partition with
    the lowest mean value.
    '''
    x = values.reshape(-1, 1)
    km = sklearn.cluster.MiniBatchKMeans(n_clusters=n_clusters, n_init=10, max_iter=100,
                                         random_state=seed)
    labels = km.fit_predict(x)
    present = np.unique(labels)
    means = [np.mean(values[labels == l]) for l in present]
    order = present[np.argsort(means)]
    new = np.zeros(len(labels), dtype=int)
    for rank, l in enumerate(order):
        new[labels == l] = rank
    return new


#############################################################################
# minimap2 / spoa / file helpers
#############################################################################

PAF_COLS = ['query_id', 'q_len', 'q_start', 'q_end', 'orientation', 'database_id',
            't_len', 't_start', 't_end', 'match', 'tot', 'mapq']
PAF_TAGS = ['tp', 'cm', 's1', 's2', 'NM', 'AS', 'ms', 'nn', 'rl', 'cg']
PAF_INT = ['q_len', 'q_start', 'q_end', 't_len', 't_start', 't_end', 'match', 'tot',
           'mapq', 'cm', 's1', 's2', 'NM', 'AS', 'ms', 'nn', 'rl']


def run_minimap2(query, database, config='-x map-ont', cigar=True, tools=None):
    '''
    Align query [id, sequence] against database [id, sequence] with minimap2.
    returns dataframe of PAF records plus match_score and similarity
    '''
    out_cols = PAF_COLS + PAF_TAGS[:-1] + ['CIGAR', 'match_score', 'similarity']
    if len(query) == 0 or len(database) == 0:
        return pd.DataFrame([], columns=out_cols)
    work = tempfile.mkdtemp(prefix='mm2_', dir=tools['workspace'])
    try:
        # minimap2 truncates ids at whitespace, so use safe temporary ids.
        # The same id must get the same name in query and database, because
        # the -D / --dual=no options compare query and target names.
        qmap = _safe_names(query['id'].values)
        dmap = _safe_names(database['id'].values)
        qfile = os.path.join(work, 'query.fa')
        dbfile = os.path.join(work, 'database.fa')
        outfile = os.path.join(work, 'results.paf')
        write_fasta(qfile, np.column_stack([list(qmap.keys()), query['sequence'].values]))
        write_fasta(dbfile, np.column_stack([list(dmap.keys()), database['sequence'].values]))
        cmd = [tools['minimap2'], '-t', str(tools['threads'])]
        if cigar:
            cmd.append('-c')
        cmd += config.split() + [dbfile, qfile, '-o', outfile]
        res = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        if res.returncode != 0:
            raise RuntimeError('minimap2 failed: ' + res.stderr.decode(errors='replace')[-2000:])
        data = parse_paf(outfile)
    finally:
        shutil.rmtree(work, ignore_errors=True)
    if len(data) == 0:
        return pd.DataFrame([], columns=out_cols)
    data['query_id'] = data['query_id'].map(qmap)
    data['database_id'] = data['database_id'].map(dmap)
    for c in PAF_INT:
        data[c] = data[c].astype(int)
    data['match_score'] = data['match'] / data['tot']
    shorter = np.minimum(data['q_len'].values, data['t_len'].values)
    data['similarity'] = np.clip(1 - data['NM'].values / shorter, 0, None)
    if not cigar:
        data = data.drop(columns=['CIGAR'])
    return data


def _safe_names(ids):
    '''map whitespace-free name -> original id (index-prefixed if not unique)'''
    names = [re.sub(r'\s+', '_', str(i)) for i in ids]
    if len(set(names)) != len(names):
        names = [str(k) + '_' + n for k, n in enumerate(names)]
    return dict(zip(names, ids))


def parse_paf(fname):
    '''Parse a minimap2 PAF file into a dataframe (missing tags -> 0).'''
    key = {t: i for i, t in enumerate(PAF_TAGS)}
    rows = []
    with open(fname) as f:
        for line in f:
            out = line.rstrip('\n').split('\t')
            if len(out) < len(PAF_COLS):
                continue
            info = [0] * len(PAF_TAGS)
            for field in out[len(PAF_COLS):]:
                parts = field.split(':')
                if parts[0] in key:
                    info[key[parts[0]]] = parts[-1]
            rows.append(out[:len(PAF_COLS)] + info)
    df = pd.DataFrame(rows, columns=PAF_COLS + PAF_TAGS)
    return df.rename(columns={'cg': 'CIGAR'})


def read_spoa(fname):
    '''Consensus = last record of spoa output, gaps removed.'''
    recs = read_fasta(fname)
    if len(recs) == 0:
        return ''
    return recs['sequence'].values[-1].replace('-', '')


def write_fasta(fname, data):
    '''data = array of [id, sequence] rows'''
    with open(fname, 'w') as f:
        for rid, seq in data:
            f.write('>' + str(rid) + '\n' + str(seq) + '\n')


def read_fasta(fname):
    '''Read a fasta file into [id, sequence].'''
    ids, seqs, cur = [], [], []
    opener = gzip.open if fname.endswith('.gz') else open
    with opener(fname, 'rt') as f:
        for line in f:
            line = line.rstrip('\n')
            if line.startswith('>'):
                if ids:
                    seqs.append(''.join(cur))
                ids.append(line[1:])
                cur = []
            elif line:
                cur.append(line.strip())
    if ids:
        seqs.append(''.join(cur))
    return pd.DataFrame({'id': ids, 'sequence': seqs})


def read_fastq(fname):
    '''Read a fastq file into [id, sequence] (qualities are not needed here).'''
    ids, seqs = [], []
    opener = gzip.open if fname.endswith('.gz') else open
    with opener(fname, 'rt') as f:
        while True:
            header = f.readline()
            if not header:
                break
            seq = f.readline().strip()
            f.readline()
            f.readline()
            if header.startswith('@'):
                ids.append(header[1:].strip())
                seqs.append(seq)
    return pd.DataFrame({'id': ids, 'sequence': seqs})


def _load_sequences(fname):
    if not os.path.isfile(fname):
        raise FileNotFoundError(fname)
    low = fname.lower()
    if re.search(r'\.(fa|fasta|fna)(\.gz)?$', low):
        df = read_fasta(fname)
    elif re.search(r'\.(fq|fastq)(\.gz)?$', low):
        df = read_fastq(fname)
    else:
        df = pd.read_csv(fname)
    if 'sequence' not in df.columns:
        raise ValueError(fname + ' needs a "sequence" column')
    df = df[df['sequence'].notna()].copy()
    df['sequence'] = df['sequence'].astype(str).str.strip().str.upper()
    df = df[df['sequence'].str.len() > 0]
    if 'id' not in df.columns:
        df['id'] = ['seq' + str(i) for i in range(len(df))]
    return df.reset_index(drop=True)


def _concat(frames):
    '''pd.concat that skips empty frames (avoids pandas FutureWarnings)'''
    keep = [f for f in frames if len(f) > 0]
    if not keep:
        return frames[0].iloc[0:0]
    return pd.concat(keep)


def _rename_clusters(df_c):
    df_c = df_c.reset_index(drop=True)
    df_c['id'] = ['cluster' + str(i) for i in range(len(df_c))]
    return df_c


def _check_tool(tool):
    if shutil.which(tool) is None:
        raise FileNotFoundError('%s not found on PATH; install it or pass its full path' % tool)


#############################################################################
# Per-sample wrapper used by the notebook
#############################################################################

def run_clustering(sample_dir, n_iter, threads=2):
    '''
    Cluster one sample folder and assign its reads to the clusters.
    sample_dir must contain <sample>_concatenated.csv; outputs go to the same folder.
    returns dict with sample, status, clusters, reads, assigned
    '''
    sample = os.path.basename(os.path.normpath(sample_dir))
    reads = os.path.join(sample_dir, sample + '_concatenated.csv')
    clusters = os.path.join(sample_dir, sample + '_clusters.csv')
    result = dict(sample=sample, status='skipped (no input)', clusters=0, reads=0, assigned=0)
    if not os.path.exists(reads):
        return result
    cluster(reads, clusters, n_iter=n_iter, threads=threads,
            log_file=os.path.join(sample_dir, sample + '_clustering.log'))
    a, counts = assign_reads(reads, clusters, threads=threads)
    result.update(status='ok', clusters=len(counts) - 1, reads=len(a),
                  assigned=int((a['status'] == 'assigned').sum()))
    return result

Overwriting clustering_tools.py


In [11]:
CLUSTER_ITER = config.getint('clustering', 'niter')
PARALLEL_SAMPLES = 4      # samples clustered at the same time
THREADS_PER_SAMPLE = 2    # minimap2/spoa threads per sample

require_tools('minimap2', 'spoa')

# import (or reload, after editing) the clustering code saved by the cell above
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()
import clustering_tools
importlib.reload(clustering_tools)

samples = qc_samples()
sample_dirs = [str(QC_ROOT / s) for s in samples]
n = len(sample_dirs)

# 'spawn' starts clean worker processes; the same behaviour on macOS and Linux
with ProcessPoolExecutor(max_workers=PARALLEL_SAMPLES,
                         mp_context=multiprocessing.get_context('spawn')) as pool:
    results = list(pool.map(clustering_tools.run_clustering, sample_dirs,
                            [CLUSTER_ITER] * n, [THREADS_PER_SAMPLE] * n))

print(f'\n{"sample":<20}{"clusters":>10}{"reads":>10}{"assigned":>10}')
for r in results:
    if r['status'] != 'ok':
        print(f'{r["sample"]:<20}{r["status"]}')
        continue
    pct = 100 * r['assigned'] / r['reads'] if r['reads'] else 0
    print(f'{r["sample"]:<20}{r["clusters"]:>10}{r["reads"]:>10}{r["assigned"]:>10}  ({pct:.1f}%)')


sample                clusters     reads  assigned
barcode01                   29      9475      9428  (99.5%)
barcode02                  100    313833    311280  (99.2%)
barcode03                   58    180251    179889  (99.8%)
barcode04                   11      5922      5916  (99.9%)


## 5. Taxonomic annotation and OTU table

Builds one ASV table and one OTU table across all samples, in five steps:

0. **Primers.** With a `[primers]` section in `config.ini`, every cluster sequence is first turned to
   the forward primer's orientation, and the primers (with anything outside them) are cut off.
   OTU clustering, BLAST and the haplotype networks then only use the region between the primers,
   so primer bases don't count as differences from the references. A primer is accepted with up
   to `max_mismatches` (4) mismatches, inserted or missing bases, within `search_window` (50) bases
   of the sequence end; IUPAC codes such as R, Y, K are allowed. Clusters with fewer than
   `min_insert` (30) bases between the primers (primer dimers) are left out. Clusters in which only
   one primer is found are trimmed on that side only, or removed with `require_both = yes`; a
   missing forward primer often means off-target amplification. If a primer occurs twice at an end
   (a primer concatemer), the inner copy is cut. Sequences in which no primer is found are turned
   the same way with `vsearch --orient`, using the others as reference, so all sequences in the
   results are in forward orientation, on the strand of the forward primer (`primer_trimming.csv`
   marks any that could not be oriented with strand `?`).
1. **ASVs.** Clusters are made per sample, so `cluster0` in one sample has nothing to do with
   `cluster0` in another. Identical sequences from all samples are merged into one ASV, with their
   reads added up; `asv_table.csv` gives the reads per sample of every ASV.
2. **OTUs across samples.** The ASVs are clustered with vsearch, at `otu_identity`, into OTUs that
   are shared across samples (the most abundant ASV is the OTU's sequence). Each OTU's read count
   per sample is the sum of the reads of that sample's clusters in the OTU.
3. **Taxonomy with BLAST**, using the functions of `haplonet.py` (which must sit next to this notebook):
   - every OTU is searched against the reference database with `blastn` (megablast by default);
   - if the best hit leaves ≥ `trim_min` bases at an OTU end unmatched (a primer concatemer or a
     chimeric piece), the OTU is trimmed to the matched part;
   - hits covering < `min_qcov` % of the (trimmed) OTU, and hits on N-runs in the reference, are
     dropped, and the best `top` hits are kept;
   - the hits at the best identity (± `tie_delta`) are pooled, and the taxonomy is the lowest rank
     at which at least `lca_cutoff` of them agree (1.0 = strict lowest common ancestor);
   - with `rank_cutoffs = yes`, ranks that the best identity doesn't support are then cleared,
     e.g. species only at ≥ 98% identity. `consensus` keeps the uncut assignment.

   The BLAST database is built once from the reference FASTA in `results/taxonomy/blastdb` and
   reused afterwards; or point `blastdb` to a database you built yourself (without `-parse_seqids`).
4. **Output.** Everything is written to `results/taxonomy/`:

| file | contents |
|---|---|
| `otu_table.csv` | one row per OTU: reads per sample, taxonomy, identity, best hit, primers found, number of ASVs, sequence |
| `asv_table.csv` | one row per ASV: its OTU, reads per sample, the OTU's taxonomy, primers found, sequence |
| `asvs.fasta` | ASV sequences, with their total reads as `;size=` |
| `primer_trimming.csv` | per sample cluster: primers found, orientation, length before and after trimming |
| `otus.fasta` | OTU sequences |
| `otu_membership.csv` | which sample cluster belongs to which ASV and OTU |
| `blast_hits.tsv` | the hits kept per OTU (used for the taxonomy and the haplotype networks) |
| `blast_search.pkl` | the same hits with their sequences, read by section 6 |

Settings come from a `[haplonet]` section in `config.ini` (the OTU identity from `[vsearch]`), with
the defaults shown below. The database defaults to `[paths] path_to_blastdb`. The identity cutoffs
per rank are common conventions, not universal values; adjust them to your marker:

```ini
[primers]
forward = GACCCTATGGAGCTTTAGAC
reverse = CCRKGGTCRCCCCAAC   ; as ordered, 5'->3' (not reverse-complemented)
max_mismatches = 4         ; mismatches/indels allowed per primer
require_both = no          ; yes = remove sequences without both primers
search_window = 50         ; primer must start within this many bases of the end
min_insert = 30            ; shorter inserts are primer dimers and left out
trim = yes                 ; no = keep the primers (default: yes if primers are given)

[vsearch]
otu_identity = 1.0
threads = 16

[haplonet]
db = MIDORI2_TOTAL_NUC_GB273_lrRNA_RAW.fasta
blastdb = MIDORI2_lrRNA_blast.fasta     ; prebuilt BLAST database prefix; empty = build it
blast_task = megablast     ; dc-megablast or blastn find more divergent hits
threads = 8
top = 100
oversample = 5
min_id = 0.70
min_qcov = 80
evalue = 1e-5
max_ambig = 0.02
trim_query = yes
trim_min = 10
tie_delta = 0.0
lca_cutoff = 0.6
rank_cutoffs = yes
species_identity = 0.98
genus_identity = 0.95
family_identity = 0.90
order_identity = 0.85
```

In [34]:
import pickle
import re
from collections import Counter
import pandas as pd

# haplonet.py (next to this notebook) does the BLAST search, the hit filtering and the
# consensus assignment here, and the alignments and networks in section 6.
if not (ROOT / 'haplonet.py').is_file():
    raise FileNotFoundError(f'haplonet.py not found in {ROOT}: put it next to the notebook')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()
import haplonet as hn
importlib.reload(hn)

RANKS = hn.RANK_ORDER          # domain, kingdom, phylum, class, order, family, genus, species


# --- settings: [vsearch] / [haplonet] in config.ini, else these defaults ----
def _get(section, key, default, conv=float):
    return conv(config.get(section, key, fallback=str(default)).strip())


OTU_IDENTITY = _get('vsearch', 'otu_identity', 1.0)   # 1.0 = OTUs are the ASVs (identical sequences)
VSEARCH_THREADS = _get('vsearch', 'threads', 8, int)

REF_FASTA = ROOT / config.get('haplonet', 'db', fallback=config.get('paths', 'path_to_blastdb'))
BLASTDB = config.get('haplonet', 'blastdb', fallback='').strip()
HN = dict(
    task=_get('haplonet', 'blast_task', 'megablast', str),
    threads=_get('haplonet', 'threads', 8, int),
    top=_get('haplonet', 'top', 100, int),
    oversample=_get('haplonet', 'oversample', 5, int),
    min_id=_get('haplonet', 'min_id', 0.70),
    min_qcov=_get('haplonet', 'min_qcov', 80),
    evalue=_get('haplonet', 'evalue', 1e-5),
    max_ambig=_get('haplonet', 'max_ambig', 0.02),
    trim=config.getboolean('haplonet', 'trim_query', fallback=True),
    trim_min=_get('haplonet', 'trim_min', 10, int),
    tie_delta=_get('haplonet', 'tie_delta', 0.0),
    lca_cutoff=_get('haplonet', 'lca_cutoff', 0.6),
)
RANK_CUTOFFS = config.getboolean('haplonet', 'rank_cutoffs', fallback=True)
RANK_IDENTITY = {                       # minimum identity needed to keep each rank
    'species': _get('haplonet', 'species_identity', 0.98),
    'genus': _get('haplonet', 'genus_identity', 0.95),
    'family': _get('haplonet', 'family_identity', 0.90),
    'order': _get('haplonet', 'order_identity', 0.85),
}
TAX_ROOT = path_results / 'taxonomy'
SEARCH_FILE = TAX_ROOT / 'blast_search.pkl'


# --- primers: orient sequences and cut the primers off ([primers] in config.ini) -----
IUPAC = {'A': 'A', 'C': 'C', 'G': 'G', 'T': 'T', 'U': 'T', 'R': 'AG', 'Y': 'CT', 'S': 'CG',
         'W': 'AT', 'K': 'GT', 'M': 'AC', 'B': 'CGT', 'D': 'AGT', 'H': 'ACT', 'V': 'ACG',
         'N': 'ACGT'}
FWD_PRIMER = config.get('primers', 'forward', fallback='').strip().upper().replace('U', 'T')
REV_PRIMER = config.get('primers', 'reverse', fallback='').strip().upper().replace('U', 'T')
for _p in (FWD_PRIMER, REV_PRIMER):
    if set(_p) - set(IUPAC):
        raise ValueError(f'[primers] {_p}: only IUPAC nucleotide codes are allowed')
TRIM_PRIMERS = config.getboolean('primers', 'trim', fallback=bool(FWD_PRIMER or REV_PRIMER))
PRIMER_MAX_MISMATCH = _get('primers', 'max_mismatches', 4, int)  # mismatches/indels allowed per primer
REQUIRE_BOTH = config.getboolean('primers', 'require_both', fallback=False)  # remove seqs lacking a primer
PRIMER_WINDOW = _get('primers', 'search_window', 50, int)  # primer must start this close to an end
MIN_INSERT = _get('primers', 'min_insert', 30, int)      # shorter inserts = primer dimers, left out


def primer_matches(primer, seq):
    """Matches of the whole primer in seq with at most PRIMER_MAX_MISMATCH mismatches or indels
    (IUPAC codes in the primer match any of their bases), as (edits, start, end) of seq[start:end],
    from left to right. Of overlapping candidate matches only the best is kept."""
    allowed = [set(IUPAC[c]) for c in primer]
    m = len(primer)
    prev, prev_start = list(range(m + 1)), [0] * (m + 1)   # column for an empty piece of seq
    hits = []
    for j, base in enumerate(seq, 1):
        cur, cur_start = [0] * (m + 1), [j] * (m + 1)       # the match may start anywhere
        for i in range(1, m + 1):
            options = ((prev[i - 1] + (base not in allowed[i - 1]), prev_start[i - 1]),   # (mis)match
                       (prev[i] + 1, prev_start[i]),                                       # extra base in seq
                       (cur[i - 1] + 1, cur_start[i - 1]))                                 # base missing in seq
            cur[i], cur_start[i] = min(options)
        if cur[m] <= PRIMER_MAX_MISMATCH:
            d, a = cur[m], cur_start[m]
            if hits and a < hits[-1][2]:        # overlaps the previous match: keep the better one
                if d < hits[-1][0]:
                    hits[-1] = (d, a, j)
            else:
                hits.append((d, a, j))
        prev, prev_start = cur, cur_start
    return hits


def trim_primers(seq):
    """Turn seq to forward orientation (the forward primer's strand) and cut off the primers and anything
    outside them. If a primer occurs twice at an end (a primer concatemer), the inner copy is cut.
    Returns (insert, status, strand); status is 'both', 'forward only', 'reverse only' or 'none'
    (no primer found: seq is returned as it is)."""
    rev_rc = hn.revcomp(REV_PRIMER) if REV_PRIMER else ''
    best = None
    for strand, s in (('+', seq), ('-', hn.revcomp(seq))):
        lo, hi, found, edits = 0, len(s), [], 0
        if FWD_PRIMER:
            matches = primer_matches(FWD_PRIMER, s[:PRIMER_WINDOW + len(FWD_PRIMER)])
            if matches:
                d, a, b = matches[-1]           # innermost copy
                lo, edits = b, edits + d
                found.append('forward')
        if rev_rc:
            off = max(0, len(s) - PRIMER_WINDOW - len(rev_rc))
            matches = [x for x in primer_matches(rev_rc, s[off:]) if off + x[1] >= lo]
            if matches:
                d, a, b = matches[0]            # innermost copy
                hi, edits = off + a, edits + d
                found.append('reverse')
        key = (len(found), -edits)
        if best is None or key > best[0]:
            best = (key, s[lo:hi], found, strand)
    _, insert, found, strand = best
    if not found:
        return seq, 'none', '+'
    status = 'both' if len(found) == 2 else f'{found[0]} only'
    return insert, status, strand


# --- OTUs across samples (vsearch) ------------------------------------------
def vsearch(*args):
    """Run vsearch; raise with its message if it fails."""
    run(['vsearch', *map(str, args), '--threads', str(VSEARCH_THREADS)])


def pool_clusters(samples, qc_root=QC_ROOT):
    """Every sample cluster with reads, turned to the forward primer's orientation and with the
    primers cut off if [primers] is set. Returns one row per cluster: sample, cluster, n_reads,
    primers, strand, length, insert_length, kept, insert. Clusters with < MIN_INSERT bases between
    the primers (primer dimers), and with require_both = yes clusters lacking a primer, get
    kept=False."""
    rows = []
    for s in samples:
        centers = pd.read_csv(qc_root / s / f'{s}_clusters.csv')
        counts = pd.read_csv(qc_root / s / f'{s}_cluster_counts.csv')
        merged = centers.merge(counts, left_on='id', right_on='cluster')
        for r in merged[merged['n_reads'] > 0].itertuples():
            if TRIM_PRIMERS:
                seq, status, strand = trim_primers(r.sequence)
                kept = len(seq) >= MIN_INSERT and (status == 'both' or not REQUIRE_BOTH)
            else:
                seq, status, strand, kept = r.sequence, 'not trimmed', '?', True
            rows.append((s, r.id, r.n_reads, status, strand, len(r.sequence), len(seq), kept, seq))
    return pd.DataFrame(rows, columns=['sample', 'cluster', 'n_reads', 'primers', 'strand',
                                       'length', 'insert_length', 'kept', 'insert'])


def orient_unprimed(members, out_dir):
    """Turn the sequences in which no primer was found to forward orientation, like the others, with
    vsearch --orient against the sequences whose orientation the primers gave. Sequences vsearch
    can't place get strand '?'."""
    unprimed = members['kept'] & (members['primers'] == 'none')
    primed = members['kept'] & ~members['primers'].isin(['none', 'not trimmed'])
    if not unprimed.any():
        return members
    if not primed.any():
        members.loc[unprimed, 'strand'] = '?'
        return members
    db, query, out, tab = (out_dir / f'orient_{f}' for f in ('db.fasta', 'query.fasta', 'out.fasta', 'tab.tsv'))
    with open(db, 'w') as fh:
        for i, seq in members.loc[primed, 'insert'].items():
            fh.write(f'>r{i}\n{seq}\n')
    with open(query, 'w') as fh:
        for i, seq in members.loc[unprimed, 'insert'].items():
            fh.write(f'>q{i}\n{seq}\n')
    run(['vsearch', '--orient', str(query), '--db', str(db), '--fastaout', str(out),
         '--tabbedout', str(tab), '--quiet'])
    for line in open(tab):
        f = line.rstrip('\n').split('\t')
        i = int(f[0][1:])
        if f[1] == '-':
            members.at[i, 'insert'] = hn.revcomp(members.at[i, 'insert'])
        members.at[i, 'strand'] = f[1] if f[1] in '+-' else '?'
    for f in (db, query, out, tab):
        f.unlink(missing_ok=True)
    return members


def dereplicate(members, out_fasta):
    """Merge identical sequences into ASVs, numbered by total reads, and write them (total reads as
    ;size=) to out_fasta. Returns (kept members with an 'asv' column, ASV table)."""
    kept = members[members['kept']].copy()
    totals = kept.groupby('insert')['n_reads'].sum().sort_values(ascending=False, kind='mergesort')
    name = {seq: f'ASV_{i + 1}' for i, seq in enumerate(totals.index)}
    kept['asv'] = kept['insert'].map(name)
    main = (kept.sort_values('n_reads', ascending=False, kind='mergesort')
            .drop_duplicates('asv').set_index('asv'))      # the ASV's most abundant cluster
    asvs = pd.DataFrame({'ASV': [name[s] for s in totals.index], 'sequence': totals.index,
                         'n_reads': totals.values})
    asvs['primers'] = asvs['ASV'].map(main['primers'])
    asvs['strand'] = asvs['ASV'].map(main['strand'])
    asvs['n_clusters'] = asvs['ASV'].map(kept['asv'].value_counts())
    asvs['length'] = asvs['sequence'].str.len()
    with open(out_fasta, 'w') as fh:
        for r in asvs.itertuples():
            fh.write(f'>{r.ASV};size={r.n_reads}\n{r.sequence}\n')
    return kept, asvs


def cluster_otus(asv_fasta, asvs, out_dir):
    """Cluster the ASVs into OTUs at otu_identity (most abundant ASV = OTU centroid; OTUs numbered
    by total reads). Returns the ASV table with 'OTU' and 'is_centroid' columns."""
    uc = out_dir / 'otus.uc'
    vsearch('--cluster_size', asv_fasta, '--id', OTU_IDENTITY, '--strand', 'both',
            '--sizein', '--uc', uc, '--quiet')
    strip = lambda lab: re.sub(r';size=\d+;?$', '', lab)
    centroid_of = {}
    for line in open(uc):
        f = line.rstrip('\n').split('\t')
        if f[0] == 'S':
            centroid_of[strip(f[8])] = strip(f[8])
        elif f[0] == 'H':
            centroid_of[strip(f[8])] = strip(f[9])
    asvs = asvs.copy()
    asvs['centroid'] = asvs['ASV'].map(centroid_of)
    totals = asvs.groupby('centroid')['n_reads'].sum().sort_values(ascending=False, kind='mergesort')
    otu_name = {c: f'OTU_{i + 1}' for i, c in enumerate(totals.index)}
    asvs['OTU'] = asvs['centroid'].map(otu_name)
    asvs['is_centroid'] = asvs['ASV'] == asvs['centroid']
    return asvs.drop(columns='centroid')


# --- taxonomy with BLAST (haplonet method) ------------------------------------
def blast_database(out_dir, log_file):
    """Prefix of the BLAST database: [haplonet] blastdb if given, otherwise one built from the
    reference FASTA in out_dir/blastdb (built once; rebuilt only if the FASTA is newer)."""
    exts = ('.nal', '.nin', '.00.nin', '.ndb')
    if BLASTDB:
        prefix = str(ROOT / BLASTDB)
        for ext in ('.nsq', '.nal', '.ndb', '.nin'):
            if prefix.endswith(ext):
                prefix = prefix[:-len(ext)]
        if not any(Path(prefix + e).exists() for e in exts):
            raise FileNotFoundError(f'No BLAST database {prefix} (give the prefix, without .nsq/.nin/...)')
        return prefix
    prefix = out_dir / 'blastdb' / 'reference'
    built = [Path(str(prefix) + e) for e in exts if Path(str(prefix) + e).exists()]
    if built and min(p.stat().st_mtime for p in built) >= REF_FASTA.stat().st_mtime:
        log.info('Reusing BLAST database %s', prefix)
        return str(prefix)
    log.info('Building BLAST database from %s (one-off, can take a few minutes)', REF_FASTA.name)
    return hn.make_blast_db(shutil.which('makeblastdb'), str(REF_FASTA), str(out_dir), str(log_file))


_REFERENCE = {}


def load_reference():
    """Reference sequences by every ID form BLAST may report (read once per session)."""
    key = (str(REF_FASTA), REF_FASTA.stat().st_mtime)
    if key not in _REFERENCE:
        log.info('Reading %s', REF_FASTA.name)
        ref_by_id = {}
        for h, s in hn.read_fasta(str(REF_FASTA)).items():
            ref_by_id[h] = (h, s)
            ref_by_id.setdefault(hn.seq_id(h), (h, s))
            ref_by_id.setdefault(h.replace('###', ' ', 1).split()[0], (h, s))   # BLAST sseqid
        _REFERENCE.clear()
        _REFERENCE[key] = ref_by_id
    return _REFERENCE[key]


def reference_taxonomy(header):
    """Lineage [(rank, name), ...], label and rank of a reference, as haplonet parses them."""
    lineage = hn.parse_taxonomy_string(hn.taxonomy_from_header(header))
    label, rank = hn.lowest_label(lineage)
    return {'lineage': lineage, 'label': label, 'rank': rank}


def consensus_lineage(lineages, cutoff):
    """haplonet's rank-by-rank consensus, returning the whole lineage: descend while the most
    common taxon at a rank is shared by >= cutoff of the hits. Returns ({rank: name}, support)."""
    dicts = []
    for lin in lineages:
        d = {r: n for r, n in lin if r in RANKS and hn.informative(n)}
        if 'species' in d and 'genus' in d and ' ' not in d['species']:
            d['species'] = f"{d['genus']} {d['species']}"
        dicts.append(d)
    out, support = dict.fromkeys(RANKS, ''), 0.0
    for rank in RANKS:
        present = [d[rank] for d in dicts if d.get(rank)]
        if not present:
            continue                        # rank not used in this database
        top, k = Counter(present).most_common(1)[0]
        if k / len(dicts) < cutoff:         # missing values count against
            break
        out[rank], support = top, k / len(dicts)
    return out, support


def blast_taxonomy(otu_seqs, otus_fasta, out_dir):
    """BLAST the OTUs and assign their taxonomy with the haplonet method.
    Returns one row per OTU; the kept hits are saved to SEARCH_FILE for section 6."""
    log_file = out_dir / 'blast.log'
    open(log_file, 'w').close()
    blastdb = blast_database(out_dir, log_file)
    ref_by_id = load_reference()

    log.info('BLAST search (%s) of %d OTUs', HN['task'], len(otu_seqs))
    by_query = hn.blast_hits(shutil.which('blastn'), str(otus_fasta), blastdb, str(out_dir),
                             HN['top'] * max(1, HN['oversample']), HN['min_id'], HN['threads'],
                             HN['task'], HN['evalue'], HN['min_qcov'], str(log_file))
    missing = {h['target'] for hs in by_query.values() for h in hs} - set(ref_by_id)
    if missing:
        raise RuntimeError(f'{len(missing)} BLAST hit IDs (e.g. {next(iter(missing))!r}) are not '
                           f'in {REF_FASTA.name}. Was the BLAST database built from this FASTA?')

    tax, search, rows = {}, {}, []
    hit_fields = ['query', 'target', 'id', 'alnlen', 'mism', 'opens', 'qcov', 'tcov', 'ql', 'tl',
                  'qstrand', 'tilo', 'tihi', 'ambig_frac', 'evalue', 'bitscore']
    with open(out_dir / 'blast_hits.tsv', 'w') as fh:
        fh.write('\t'.join(hit_fields) + '\n')
        for otu, seq in otu_seqs.items():
            # trim primer artefacts / chimeric ends, drop hits with low query cover
            hits, lo, hi = hn.fit_hits_to_query(by_query.get(otu, []), len(seq), HN['min_qcov'],
                                                HN['trim'], HN['trim_min'])
            # drop hits on N-runs / ambiguous reference regions, keep the best `top`
            kept, dropped = [], 0
            for h in hits:
                s = ref_by_id[h['target']][1][int(h['tilo']) - 1:int(h['tihi'])]
                h['seg'] = s.upper().replace('U', 'T')
                h['ambig_frac'] = round(sum(c not in 'ACGT' for c in h['seg']) / max(1, len(h['seg'])), 4)
                if h['ambig_frac'] > HN['max_ambig']:
                    dropped += 1
                else:
                    kept.append(h)
            kept = kept[:HN['top']]
            for h in kept:
                fh.write('\t'.join(str(h[f]) for f in hit_fields) + '\n')
                if h['target'] not in tax:
                    tax[h['target']] = reference_taxonomy(ref_by_id[h['target']][0])
            trim = (lo, hi, len(seq)) if (lo, hi) != (1, len(seq)) else None
            a = hn.consensus_assignment(kept, tax, HN['tie_delta'], HN['lca_cutoff'])
            search[otu] = dict(seq=seq[lo - 1:hi], trim=trim, hits=kept, assign=a, n_dropped=dropped)

            row = dict(OTU=otu, n_hits=len(kept), n_ambiguous_hits_dropped=dropped,
                       query_region=f'{lo}-{hi}', trimmed_bp=len(seq) - (hi - lo + 1))
            if not kept:
                rows.append(dict(row, assigned_rank='unassigned', consensus='no hits'))
                continue
            tied = [h for h in kept if h['id'] >= a['identity'] - HN['tie_delta']]
            taxa, support = consensus_lineage([tax[h['target']]['lineage'] for h in tied],
                                              HN['lca_cutoff'])
            if RANK_CUTOFFS:            # drop ranks the identity doesn't support
                for rank, cutoff in RANK_IDENTITY.items():
                    if a['identity'] / 100 < cutoff:
                        for lower in RANKS[RANKS.index(rank):]:
                            taxa[lower] = ''
            deepest = [r for r in RANKS if taxa[r]]
            rows.append(dict(row, **taxa,
                             assigned_rank=deepest[-1] if deepest else 'unassigned',
                             identity=a['identity'], query_cov=kept[0]['qcov'],
                             n_top_hits=a['n_top_hits'], support=round(support, 2),
                             consensus=f"{a['label']} ({a['rank']})",
                             best_hit=hn.seq_id(kept[0]['target']),
                             taxa_best_hits='; '.join(f'{k} ({v})' for k, v in
                                                      a['top_labels'].most_common())))

    with open(SEARCH_FILE, 'wb') as fh:
        pickle.dump(dict(settings=HN, ref_fasta=str(REF_FASTA), otus=search, tax=tax), fh)
    n_trim = sum(1 for o in search.values() if o['trim'])
    if n_trim:
        log.info('%d OTUs trimmed to the part covered by their best BLAST hit (see query_region)', n_trim)
    return pd.DataFrame(rows)


def build_otu_table(samples, qc_root=QC_ROOT, out_dir=TAX_ROOT):
    """OTU table with reads per sample and taxonomy; also writes the ASV table and the
    supporting files."""
    out_dir.mkdir(parents=True, exist_ok=True)
    members = pool_clusters(samples, qc_root)
    if members.empty:
        raise RuntimeError('No clusters with assigned reads: run section 4 first')
    members = orient_unprimed(members, out_dir)
    members.drop(columns='insert').to_csv(out_dir / 'primer_trimming.csv', index=False)
    if not members['kept'].any():
        raise RuntimeError('No cluster passed the primer filter: check [primers] in config.ini')

    # identical sequences -> ASVs -> OTUs
    asv_fasta = out_dir / 'asvs.fasta'
    members, asvs = dereplicate(members, asv_fasta)
    asvs = cluster_otus(asv_fasta, asvs, out_dir)
    members['otu'] = members['asv'].map(asvs.set_index('ASV')['OTU'])
    members[['sample', 'cluster', 'asv', 'otu', 'n_reads', 'primers', 'strand']].to_csv(
        out_dir / 'otu_membership.csv', index=False)

    centroids = asvs[asvs['is_centroid']].set_index('OTU')
    order = sorted(centroids.index, key=lambda o: int(o.split('_')[1]))
    otu_seqs = {o: centroids.at[o, 'sequence'] for o in order}
    otus_fasta = out_dir / 'otus.fasta'
    with open(otus_fasta, 'w') as fh:
        for otu, seq in otu_seqs.items():
            fh.write(f'>{otu}\n{seq}\n')

    counts = members.pivot_table(index='otu', columns='sample', values='n_reads',
                                 aggfunc='sum', fill_value=0)
    counts = counts.reindex(index=list(otu_seqs), columns=samples, fill_value=0)
    counts['total_reads'] = counts.sum(axis=1)
    taxonomy = blast_taxonomy(otu_seqs, otus_fasta, out_dir).set_index('OTU').reindex(counts.index)
    taxonomy['assigned_rank'] = taxonomy['assigned_rank'].fillna('unassigned')
    taxonomy[RANKS] = taxonomy.reindex(columns=RANKS).fillna('')
    table = counts.join(taxonomy)
    table['primers'] = centroids['primers'].reindex(table.index)
    table['n_asvs'] = asvs['OTU'].value_counts().reindex(table.index)
    table['sequence'] = [otu_seqs[o] for o in table.index]
    table.index.name = 'OTU'
    table.to_csv(out_dir / 'otu_table.csv')

    # ASV table: reads per sample, with the OTU and the OTU's taxonomy
    asv_counts = members.pivot_table(index='asv', columns='sample', values='n_reads',
                                     aggfunc='sum', fill_value=0)
    asv_counts = asv_counts.reindex(index=asvs['ASV'], columns=samples, fill_value=0)
    asv_counts['total_reads'] = asv_counts.sum(axis=1)
    asv_table = (asvs.set_index('ASV')[['OTU']].join(asv_counts)
                 .join(table[['assigned_rank', 'consensus'] + RANKS], on='OTU')
                 .join(asvs.set_index('ASV')[['primers', 'strand', 'n_clusters', 'length', 'sequence']]))
    asv_table.index.name = 'ASV'
    asv_table.to_csv(out_dir / 'asv_table.csv')
    return table


In [35]:
# Check that the reference headers are parsed correctly before running the search
require_tools('vsearch', 'blastn', 'makeblastdb')
if not REF_FASTA.is_file():
    raise FileNotFoundError(f'Reference database not found: {REF_FASTA}')
print('Database:', REF_FASTA)
if TRIM_PRIMERS:
    print(f'Primers: forward {FWD_PRIMER}, reverse {REV_PRIMER} '
          f'(max {PRIMER_MAX_MISMATCH} mismatches, cut off before OTU clustering)')
    print('Sequences without both primers:', 'removed' if REQUIRE_BOTH else 'kept')
else:
    print('Primers: not trimmed (no [primers] in config.ini)')
headers = []
with open(REF_FASTA) as fh:
    for line in fh:
        if line.startswith('>'):
            headers.append(line[1:].strip())
            if len(headers) == 3:
                break
for header in headers:
    t = reference_taxonomy(header)
    print(f'\n{header[:120]}...\n  -> {hn.seq_id(header)}   label: {t["label"]} ({t["rank"]})')
    ranked = {r: n for r, n in t['lineage'] if r in RANKS}
    for rank in RANKS:
        print(f'     {rank:<8} {ranked.get(rank, "")}')

Database: /Users/pascalhabluetzel/Documents/ANERIS_workshop_Crete_2026/grifo/v_0.7/MIDORI2_lrRNA_blast.fasta
Primers: forward GACCCTATGGAGCTTTAGAC, reverse CCRKGGTCRCCCCAAC (max 4 mismatches, cut off before OTU clustering)
Sequences without both primers: kept

LC790133.1.965.>1079 root_1;cellularroot_cellular_organisms_131567;domain_Eukaryota_2759;clade_Opisthokonta_33154;kingdo...
  -> LC790133.1.965.>1079   label: Chaunax penicillatus (species)
     domain   Eukaryota
     kingdom  Metazoa
     phylum   Chordata
     class    Actinopteri
     order    Lophiiformes
     family   Chaunacidae
     genus    Chaunax
     species  Chaunax penicillatus

AB808683.1.<1.>523 root_1;cellularroot_cellular_organisms_131567;domain_Eukaryota_2759;clade_Opisthokonta_33154;kingdom_...
  -> AB808683.1.<1.>523   label: Gymnothorax sagmacephalus (species)
     domain   Eukaryota
     kingdom  Metazoa
     phylum   Chordata
     class    Actinopteri
     order    Anguilliformes
     family   Muraenidae
 

In [36]:
samples = qc_samples()
otu_table = build_otu_table(samples)

pt = pd.read_csv(TAX_ROOT / 'primer_trimming.csv')
n_asv = len(pd.read_csv(TAX_ROOT / 'asv_table.csv'))
print(f'{len(samples)} samples: {pt["kept"].sum()} sample clusters -> {n_asv} unique sequences (ASVs) '
      f'-> {len(otu_table)} OTUs (ASVs clustered at {OTU_IDENTITY:.0%})')
if TRIM_PRIMERS:
    n_unknown = int((pt['kept'] & (pt['strand'] == '?')).sum())
    print('Orientation: all sequences in forward orientation' if not n_unknown else
          f"Orientation: {n_unknown} sequences without primers could not be oriented (strand '?')")
if TRIM_PRIMERS:
    short = pt['insert_length'] < MIN_INSERT
    pt.loc[~pt['kept'] & ~short, 'primers'] += ' (removed)'
    pt.loc[~pt['kept'] & short, 'primers'] = f'insert < {MIN_INSERT} bp (removed)'
    summary = pt.groupby('primers').agg(clusters=('cluster', 'size'), reads=('n_reads', 'sum'))
    summary['% reads'] = (100 * summary['reads'] / summary['reads'].sum()).round(1)
    print(f'\nPrimers found in the sample clusters (max {PRIMER_MAX_MISMATCH} mismatches):')
    print(summary.sort_values('reads', ascending=False).to_string())
print(f'Taxonomy: BLAST {HN["task"]}, consensus of the best hits (>= {HN["lca_cutoff"]:.0%} agreement)'
      + (', ranks cut by identity' if RANK_CUTOFFS else ''))
print('\nOTUs per assigned rank:')
print(otu_table['assigned_rank'].value_counts().reindex(RANKS + ['unassigned']).dropna().astype(int).to_string())
print(f'\nOTUs trimmed (primer artefact / chimera): {(otu_table["trimmed_bp"] > 0).sum()}')
print(f'\nReads in the OTU table per sample:')
print(otu_table[samples].sum().to_string())
print(f'\nWritten to {TAX_ROOT}')
otu_table.drop(columns='sequence').head(20)

2026-11-11 11:46:05,606 INFO Reading MIDORI2_lrRNA_blast.fasta
2026-11-11 11:46:08,379 INFO BLAST search (megablast) of 124 OTUs
  $ /Users/pascalhabluetzel/miniforge3/envs/grifo2/bin/blastn -task megablast -query /Users/pascalhabluetzel/Documents/ANERIS_workshop_Crete_2026/grifo/v_0.7/results/taxonomy/otus.fasta -db /Users/pascalhabluetzel/Documents/ANERIS_workshop_Crete_2026/grifo/v_0.7/MIDORI2_lrRNA -max_target_seqs 500 -max_hsps 1 -perc_identity 70.0 -evalue 1e-05 -num_threads 8 -outfmt 6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore score qlen slen qcovhsp qseq sseq -out /Users/pascalhabluetzel/Documents/ANERIS_workshop_Crete_2026/grifo/v_0.7/results/taxonomy/blast_hits_raw.tsv


4 samples: 198 sample clusters -> 144 unique sequences (ASVs) -> 124 OTUs (ASVs clustered at 100%)
Orientation: all sequences in forward orientation

Primers found in the sample clusters (max 4 mismatches):
              clusters   reads  % reads
primers                                
both               114  317173     62.6
reverse only        84  189340     37.4
Taxonomy: BLAST megablast, consensus of the best hits (>= 60% agreement), ranks cut by identity

OTUs per assigned rank:
assigned_rank
kingdom         1
species        18
unassigned    105

OTUs trimmed (primer artefact / chimera): 0

Reads in the OTU table per sample:
barcode01      9428
barcode02    311280
barcode03    179889
barcode04      5916

Written to /Users/pascalhabluetzel/Documents/ANERIS_workshop_Crete_2026/grifo/v_0.7/results/taxonomy


,barcode01,barcode02,barcode03,barcode04,total_reads,n_hits,n_ambiguous_hits_dropped,query_region,trimmed_bp,domain,...,assigned_rank,identity,query_cov,n_top_hits,support,consensus,best_hit,taxa_best_hits,primers,n_asvs
OTU,,,,,,,,,,,,,,,,,,,,,
OTU_1,0,53392,0,0,53392,68,0,1-72,0,Eukaryota,...,species,100.0,100.0,12.0,1.00,Chelon auratus (species),AY169698.1.<1.>563,Chelon auratus (12),both,1
OTU_2,0,51804,0,0,51804,0,0,1-155,0,,...,unassigned,NaN,NaN,NaN,NaN,no hits,NaN,NaN,both,1
OTU_3,0,0,44790,0,44790,0,0,1-182,0,,...,unassigned,NaN,NaN,NaN,NaN,no hits,NaN,NaN,reverse only,1
OTU_4,0,0,40384,0,40384,0,0,1-182,0,,...,unassigned,NaN,NaN,NaN,NaN,no hits,NaN,NaN,reverse only,1
OTU_5,0,0,29594,0,29594,0,0,1-402,0,,...,unassigned,NaN,NaN,NaN,NaN,no hits,NaN,NaN,both,1
OTU_6,0,27656,0,0,27656,9,0,1-71,0,Eukaryota,...,species,100.0,100.0,5.0,1.00,Dicentrarchus labrax (species),KJ128761.1.<1.>554,Dicentrarchus labrax (5),both,1
OTU_7,0,0,27373,0,27373,0,0,1-402,0,,...,unassigned,NaN,NaN,NaN,NaN,no hits,NaN,NaN,both,1
OTU_8,0,27124,0,0,27124,9,0,1-72,0,Eukaryota,...,species,100.0,100.0,8.0,1.00,Sparus aurata (species),KT805959.1.1090.2785,Sparus aurata (8),both,1
OTU_9,3725,2992,16183,3642,26542,100,0,1-69,0,Eukaryota,...,kingdom,100.0,100.0,100.0,0.85,Metazoa (kingdom),JF339212.1.<1.>592,Tapinoma indicum (28); Homo sapiens (19); Acon...,both,1


## 6. Haplotype networks

For every OTU, `haplonet.py` aligns the OTU with the BLAST hits kept in section 5 (only the part of
each reference that matches the OTU), trims the alignment to the OTU, collapses identical sequences
into haplotypes and builds a minimum spanning network. Haplotypes are labelled with their taxonomy.

All networks go into one page, `results/haplotype_networks/haplotype_networks.html`: pick an OTU from
the dropdown, filter the list by ID or taxon, or step through the OTUs with the arrow keys. The page
shows each OTU's read count, the OTU-table taxonomy and the consensus of its best hits. Below the
network, an NCBI-BLAST-style alignment shows the OTU against its best hit; pick another of the top
hits from the list above it, or click a node to see the alignment of that haplotype.
If the best hit is a eukaryote identified to species, the page links to its WoRMS page, its OBIS
distribution map and its GBIF occurrence map. Their IDs are looked up online once per species and
kept in `results/haplotype_networks/taxon_links.json`; without internet, or for names WoRMS or
GBIF don't know, the links search for the name instead (marked *search*). It loads
vis-network from a CDN, so viewing it needs internet. The alignments per OTU are in
`results/haplotype_networks/otus/<OTU>/`.

This section reads `results/taxonomy/otu_table.csv` and `blast_search.pkl`, so it doesn't need
section 5 to have run in this session (only its first code cell, which loads the settings).
The alignments are stored by section 5: results from before this feature need section 5 rerun.
Optional settings in `[haplonet]`:

```ini
network_min_reads = 0     ; skip OTUs with fewer reads
mafft_mode = auto         ; auto, localpair, globalpair or genafpair
gaps = fifth              ; gaps as fifth state, missing data, or remove gap columns
max_label_taxa = 2        ; taxa named in a node label before '+N more'
max_alignments = 10       ; top hits whose BLAST alignment is shown below the network
taxon_links = yes         ; look up WoRMS/GBIF IDs online for the best-hit links (no = name search links)
title = GRIFO haplotype networks
```

In [37]:
NETWORK_MIN_READS = _get('haplonet', 'network_min_reads', 0, int)
MAFFT_MODE = _get('haplonet', 'mafft_mode', 'auto', str)
GAPS = _get('haplonet', 'gaps', 'fifth', str)
MAX_LABEL_TAXA = _get('haplonet', 'max_label_taxa', 2, int)
NETWORK_TITLE = _get('haplonet', 'title', 'GRIFO haplotype networks', str)
MAX_ALIGNMENTS = _get('haplonet', 'max_alignments', 10, int)   # top hits with a BLAST alignment in the page
TAXON_LINKS = config.getboolean('haplonet', 'taxon_links', fallback=True)  # look up WoRMS/GBIF IDs online
NET_ROOT = path_results / 'haplotype_networks'


def otu_table_taxonomy(row):
    """'name (rank), xx% id' from the OTU table's taxonomy columns."""
    deepest = [r for r in RANKS if isinstance(row.get(r), str) and row[r]]
    if not deepest:
        return 'unassigned'
    txt = f'{row[deepest[-1]]} ({deepest[-1]})'
    if pd.notna(row.get('identity')):
        txt += f', {row["identity"]:.1f}% id'
    return txt


def build_networks(otu_table, search, out_dir=NET_ROOT, workers=HN['threads']):
    """Align, build and draw a haplotype network per OTU; returns the path of the HTML page."""
    out_dir.mkdir(parents=True, exist_ok=True)
    mafft = shutil.which('mafft')
    otus = [o for o in otu_table.index
            if o in search['otus'] and otu_table.loc[o, 'total_reads'] >= NETWORK_MIN_READS]

    jobs = []
    for otu in otus:
        s = search['otus'][otu]
        hit_seqs = []
        for k, h in enumerate(s['hits']):
            seq = hn.revcomp(h['seg']) if h.get('qstrand') == '-' else h['seg']
            name = f'H{k + 1:03d}_' + re.sub(r'[^\w.\-]', '_', hn.seq_id(h['target']))[:60]
            hit_seqs.append((name, seq))
        jobs.append(dict(otu=otu, qname='QUERY_' + otu, qseq=s['seq'], hit_seqs=hit_seqs,
                         outdir=str(out_dir / 'otus' / re.sub(r'[^\w.\-]', '_', otu)),
                         mafft=mafft, mafft_mode=MAFFT_MODE, gaps=GAPS, drop_query_gaps=False))

    log.info('Aligning and building %d networks (%d workers)', len(jobs), workers)
    results = {}
    with ProcessPoolExecutor(max_workers=workers,
                             mp_context=multiprocessing.get_context('spawn')) as pool:
        for job, res in zip(jobs, pool.map(hn.process_otu, jobs)):
            results[job['otu']] = res

    # page data, as haplonet.py writes it
    targets, t_index, labels, l_index = [], {}, [], {}

    def tidx(target):
        if target not in t_index:
            t = search['tax'][target]
            if t['label'] not in l_index:
                l_index[t['label']] = len(labels)
                labels.append(t['label'])
            t_index[target] = len(targets)
            lineage = '; '.join(n for r, n in t['lineage'] if r is None or r in RANKS)
            targets.append([hn.seq_id(target), lineage, l_index[t['label']]])
        return t_index[target]

    # links to WoRMS, OBIS and GBIF for best hits that are eukaryote species
    link_name = {}
    for otu in otus:
        bh = hn.best_hit(search['otus'][otu]['hits'])
        link_name[otu] = hn.link_species(search['tax'][bh['target']]) if bh else None
    links = hn.taxon_links(link_name.values(), str(out_dir / 'taxon_links.json'),
                           online=TAXON_LINKS, log=log.warning)

    page_otus, n_err = [], 0
    for otu in otus:
        s, res, row = search['otus'][otu], results[otu], otu_table.loc[otu]
        hits, a = s['hits'], s['assign']
        notes = []
        if not hits:
            notes.append(f'No usable hits >= {search["settings"]["min_id"]:.0%} identity')
        if s['trim']:
            lo, hi, L0 = s['trim']
            cut = ', '.join(x for x in (f'1–{lo - 1}' if lo > 1 else '',
                                        f'{hi + 1}–{L0}' if hi < L0 else '') if x)
            notes.append(f'Bases {cut} of {L0} not matched by the best hit were removed; '
                         f'analysed region {lo}–{hi} (possible primer artefact or chimera)')
        if s['n_dropped']:
            notes.append(f'{s["n_dropped"]} hits on N-runs/ambiguous reference regions discarded')
        if 'error' in res:
            n_err += 1
            notes.append(res['error'])
            res = {'haps': [[-1] + list(range(len(hits)))], 'edges': [], 'L': len(s['seq'])}
        nodes = [{'m': [[-1, 100.0] if x < 0 else [tidx(hits[x]['target']), hits[x]['id']]
                        for x in members]} for members in res['haps']]
        if a:
            taxa = ', '.join(f'{k} ({v})' for k, v in a['top_labels'].most_common(6))
            if len(a['top_labels']) > 6:
                taxa += ', …'
            assign = dict(label=a['label'], rank=a['rank'], support=a['support'],
                          identity=a['identity'], n=a['n_top_hits'], taxa=taxa)
        else:
            assign = dict(label='no hits', rank='')
        aln = hn.alignment_records(hits, lambda t: search['tax'][t]['label'], MAX_ALIGNMENTS)
        for rec in aln:
            rec['t'] = tidx(rec.pop('target'))
        page_otus.append(dict(id=otu, len=len(s['seq']),
                              len_full=s['trim'][2] if s['trim'] else None,
                              reads=int(row['total_reads']), table_tax=otu_table_taxonomy(row),
                              assign=assign, aln=aln, nhits=len(hits), L=res['L'], nodes=nodes,
                              edges=res['edges'], links=links.get(link_name[otu]),
                              note='; '.join(notes) or None))

    html_path = out_dir / 'haplotype_networks.html'
    hn.write_html(str(html_path), NETWORK_TITLE,
                  {'otus': page_otus, 'targets': targets, 'labels': labels}, MAX_LABEL_TAXA)
    if n_err:
        log.warning('%d OTUs failed to align; see the notes on their page', n_err)
    return html_path

In [38]:
require_tools('mafft')
otu_table = pd.read_csv(TAX_ROOT / 'otu_table.csv', index_col='OTU', keep_default_na=False,
                        na_values=[''])
with open(SEARCH_FILE, 'rb') as fh:
    search = pickle.load(fh)

html_path = build_networks(otu_table, search)
print(f'Haplotype networks for {sum(otu_table["total_reads"] >= NETWORK_MIN_READS)} OTUs')
print(f'Page: {html_path}')

2026-11-11 11:46:09,184 INFO Aligning and building 124 networks (8 workers)


Haplotype networks for 124 OTUs
Page: /Users/pascalhabluetzel/Documents/ANERIS_workshop_Crete_2026/grifo/v_0.7/results/haplotype_networks/haplotype_networks.html
